In [ ]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
import pickle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp '/content/drive/MyDrive/AML_project/aml_synthetic_dataset_v2.csv' '/content/'

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/AML_project/aml_synthetic_dataset_v2.csv")

print(df.head())

  transaction_id  sender_id receiver_id   amount    country transaction_type  \
0          TXN_0  CUST_3619   CUST_5457   938.54      India          deposit   
1          TXN_1  CUST_4654   CUST_7234  6020.24      India          payment   
2          TXN_2  CUST_3069   CUST_7466  2633.49      India       withdrawal   
3          TXN_3  CUST_1902   CUST_6840  1825.89  Singapore          deposit   
4          TXN_4  CUST_1653   CUST_7860   339.25         UK          deposit   

  mode_of_payment            timestamp  is_high_risk_country  is_large_txn  \
0     credit_card  2024-02-20 19:18:00                     0             0   
1     credit_card  2024-02-04 02:43:00                     0             0   
2             upi  2024-09-12 08:16:00                     0             0   
3             upi  2024-01-03 08:47:00                     0             0   
4      debit_card  2024-03-19 09:26:00                     0             0   

   is_suspicious  
0              0  
1           

In [ ]:
df.shape

(20000, 11)

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["hour"]  = df["timestamp"].dt.hour
df["day"]   = df["timestamp"].dt.day
df["month"] = df["timestamp"].dt.month


In [ ]:
df = df.drop(columns=["transaction_id"])

In [ ]:
df["is_large_txn"]         = (df["amount"] > 8000).astype(int)
df["is_high_risk_country"] = df["country"].isin(["Nigeria", "UAE"]).astype(int)

In [ ]:
df["sender_txn_count"]        = df.groupby("sender_id")["amount"].transform("count")
df["sender_avg_amount"]       = df.groupby("sender_id")["amount"].transform("mean")
df["sender_unique_receivers"] = df.groupby("sender_id")["receiver_id"].transform("nunique")

df = df.sort_values(["sender_id", "timestamp"])
df["time_diff"] = df.groupby("sender_id")["timestamp"].diff().dt.seconds.fillna(0)

In [ ]:
sender_stats = df.groupby("sender_id").agg(
    sender_txn_count        = ("amount",      "count"),
    sender_avg_amount       = ("amount",      "mean"),
    sender_unique_receivers = ("receiver_id", "nunique")
).reset_index()

print("Sender stats sample:")
print(sender_stats.describe())

Sender stats sample:
       sender_txn_count  sender_avg_amount  sender_unique_receivers
count       3977.000000        3977.000000              3977.000000
mean           5.028916        2002.638474                 5.025145
std            2.203418        1036.663698                 2.199865
min            1.000000          15.970000                 1.000000
25%            3.000000        1308.602500                 3.000000
50%            5.000000        1835.695714                 5.000000
75%            6.000000        2512.445000                 6.000000
max           15.000000       11325.080000                15.000000


In [ ]:
y = df["is_suspicious"]

X = df.drop(columns=[
    "is_suspicious",
    "sender_id",
    "receiver_id",
    "timestamp"
])

In [ ]:
X = pd.get_dummies(X)

print(f"\nFeature count: {X.shape[1]}")
print("Features:", list(X.columns))


Feature count: 26
Features: ['amount', 'is_high_risk_country', 'is_large_txn', 'hour', 'day', 'month', 'sender_txn_count', 'sender_avg_amount', 'sender_unique_receivers', 'time_diff', 'country_India', 'country_Nigeria', 'country_Singapore', 'country_UAE', 'country_UK', 'country_USA', 'transaction_type_deposit', 'transaction_type_payment', 'transaction_type_transfer', 'transaction_type_withdrawal', 'mode_of_payment_cash', 'mode_of_payment_credit_card', 'mode_of_payment_crypto', 'mode_of_payment_debit_card', 'mode_of_payment_upi', 'mode_of_payment_wire_transfer']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.01,
    scale_pos_weight=scale_pos_weight,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=600, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred))

# ROC-AUC
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")



Threshold: 0.5
              precision    recall  f1-score   support

           0       0.97      0.84      0.90      3854
           1       0.09      0.42      0.15       146

    accuracy                           0.82      4000
   macro avg       0.53      0.63      0.53      4000
weighted avg       0.94      0.82      0.87      4000


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.98      0.74      0.84      3854
           1       0.08      0.58      0.14       146

    accuracy                           0.73      4000
   macro avg       0.53      0.66      0.49      4000
weighted avg       0.95      0.73      0.82      4000


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.98      0.67      0.80      3854
           1       0.08      0.71      0.14       146

    accuracy                           0.67      4000
   macro avg       0.53      0.69      0.47      4000
weighted avg       0.95   

In [ ]:
with open("aml_pipeline.pkl", "wb") as f:
    pickle.dump({
        "model":        model,
        "columns":      list(X_train.columns),
        "sender_stats": sender_stats
    }, f)